# Download do Dataset LUNA16

Notebook para baixar e organizar o dataset LUNA16 a partir do Zenodo.

**Fonte dos dados:**
- Part 1: https://zenodo.org/records/3723295
- Part 2: https://zenodo.org/records/4121926

O download suporta **resume** automatico: se a conexao cair, basta rodar novamente que o download continua de onde parou. Arquivos ja existentes em `data/luna/` sao ignorados.

In [ ]:
import os
import zipfile
import shutil
from pathlib import Path

import requests
from tqdm import tqdm

## Configuracao

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
LUNA_DIR = PROJECT_ROOT / "data" / "luna"

RAW_DIR.mkdir(parents=True, exist_ok=True)
LUNA_DIR.mkdir(parents=True, exist_ok=True)

ZENODO_PART1 = "https://zenodo.org/records/3723295/files"
ZENODO_PART2 = "https://zenodo.org/records/4121926/files"

LUNA16_FILES = {
    # CSVs (Part 1)
    "annotations.csv": f"{ZENODO_PART1}/annotations.csv?download=1",
    "candidates.csv":  f"{ZENODO_PART1}/candidates.csv?download=1",
    # Subsets (Part 1)
    "subset0.zip": f"{ZENODO_PART1}/subset0.zip?download=1",
    "subset1.zip": f"{ZENODO_PART1}/subset1.zip?download=1",
    "subset2.zip": f"{ZENODO_PART1}/subset2.zip?download=1",
    "subset3.zip": f"{ZENODO_PART1}/subset3.zip?download=1",
    "subset4.zip": f"{ZENODO_PART1}/subset4.zip?download=1",
    "subset5.zip": f"{ZENODO_PART1}/subset5.zip?download=1",
    "subset6.zip": f"{ZENODO_PART1}/subset6.zip?download=1",
    # Subsets (Part 2)
    "subset7.zip": f"{ZENODO_PART2}/subset7.zip?download=1",
    "subset8.zip": f"{ZENODO_PART2}/subset8.zip?download=1",
    "subset9.zip": f"{ZENODO_PART2}/subset9.zip?download=1",
}

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RAW_DIR: {RAW_DIR}")
print(f"LUNA_DIR: {LUNA_DIR}")
print(f"Total de arquivos: {len(LUNA16_FILES)}")

## Funcao de download com resume

In [ ]:
def download_file(url: str, dest: Path, chunk_size: int = 8192) -> Path:
    """Baixa um arquivo com suporte a resume.

    Se o arquivo ja existe parcialmente em `dest`, continua o download
    de onde parou usando o header Range.

    Returns:
        Path do arquivo baixado.
    """
    dest.parent.mkdir(parents=True, exist_ok=True)

    # Bytes ja baixados (para resume)
    downloaded = dest.stat().st_size if dest.exists() else 0

    headers = {}
    if downloaded > 0:
        headers["Range"] = f"bytes={downloaded}-"

    response = requests.get(url, headers=headers, stream=True, timeout=30)

    # Se o servidor retorna 416, o arquivo ja esta completo
    if response.status_code == 416:
        print(f"  Ja completo: {dest.name}")
        return dest

    response.raise_for_status()

    # Tamanho total do arquivo
    if response.status_code == 206:  # Partial Content (resume)
        content_range = response.headers.get("Content-Range", "")
        total = int(content_range.split("/")[-1]) if "/" in content_range else None
    else:
        total = int(response.headers.get("Content-Length", 0)) or None
        downloaded = 0  # servidor nao suporta resume, recomecar

    mode = "ab" if response.status_code == 206 else "wb"

    if downloaded > 0:
        print(f"  Resumindo {dest.name} a partir de {downloaded / 1e6:.1f} MB...")

    progress = tqdm(
        total=total,
        initial=downloaded,
        unit="B",
        unit_scale=True,
        desc=dest.name,
    )

    with open(dest, mode) as f:
        for chunk in response.iter_content(chunk_size=chunk_size):
            f.write(chunk)
            progress.update(len(chunk))

    progress.close()
    return dest

## Funcao para descompactar e organizar

In [ ]:
def extract_to_luna(zip_path: Path) -> None:
    """Extrai um zip de subset para data/luna/ e remove o zip de raw/."""
    print(f"  Extraindo {zip_path.name} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(LUNA_DIR)
    print(f"  Extraido. Removendo {zip_path.name} de raw/")
    zip_path.unlink()

## Funcao: download e extracao

In [ ]:
def download_and_extract(file_name: str, url: str, raw_dir: Path, luna_dir: Path) -> None:
    """Baixa e organiza um unico arquivo do LUNA16.

    Se o arquivo final ja existe em data/luna/, pula o download.
    Se e um zip, extrai para data/luna/ e remove o zip.

    Args:
        file_name: Nome do arquivo (ex: 'subset0.zip', 'annotations.csv')
        url: URL de download do arquivo
        raw_dir: Diretorio para downloads temporarios
        luna_dir: Diretorio final dos dados
    """
    is_zip = file_name.endswith(".zip")

    # Verificar se ja existe em luna/
    if is_zip:
        subset_name = file_name.replace(".zip", "")
        subset_dir = luna_dir / subset_name
        if subset_dir.exists() and any(subset_dir.iterdir()):
            print(f"  [OK] {subset_name}/ ja existe em luna/")
            raw_zip = raw_dir / file_name
            if raw_zip.exists():
                raw_zip.unlink()
            return
    else:
        luna_file = luna_dir / file_name
        if luna_file.exists():
            print(f"  [OK] {file_name} ja existe em luna/")
            raw_file = raw_dir / file_name
            if raw_file.exists():
                raw_file.unlink()
            return

    # Baixar para raw/
    raw_path = raw_dir / file_name
    download_file(url, raw_path)

    # Mover ou extrair para luna/
    if is_zip:
        extract_to_luna(raw_path)
    else:
        dest = luna_dir / file_name
        shutil.move(str(raw_path), str(dest))
        print(f"  Movido para data/luna/{file_name}")

## Teste: baixar um unico subset

Descomente a linha abaixo para testar o download de um subset individual.

In [ ]:
# download_and_extract("subset0.zip", LUNA16_FILES["subset0.zip"], RAW_DIR, LUNA_DIR)

## Download do dataset completo

Execute a celula abaixo para baixar todos os 10 subsets e os CSVs. O download total e de aproximadamente 67 GB. Arquivos ja presentes em `data/luna/` serao automaticamente ignorados.

In [ ]:
for file_name, url in LUNA16_FILES.items():
    print(f"\nProcessando: {file_name}")
    try:
        download_and_extract(file_name, url, RAW_DIR, LUNA_DIR)
    except Exception as e:
        print(f" [ERRO] {file_name}: {e}")
        print(f"  Re-execute esta celula para tentar novamente.")